## Step 2: Install Dependencies

In [ ]:
# Install ultralytics
!pip install ultralytics -q

## Step 3: Upload Dataset

**Option A: From Google Drive** (if dataset is already there)
**Option B: Direct upload** (use Files icon in sidebar)

In [ ]:
# Option A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy dataset from Drive (adjust path as needed):
# !cp -r /content/drive/MyDrive/SDP/datasets/pallet_v1 /content/

# OR Option B: Upload directly, then unzip if needed:
# !unzip -q /content/pallet_v1.zip -d /content/

## Step 4: Fix data.yaml Path

The data.yaml file needs to be updated for Colab paths.

In [ ]:
# Fix data.yaml path for Colab
import yaml
from pathlib import Path

data_yaml_path = "/content/pallet_v1/data.yaml"

if Path(data_yaml_path).exists():
    # Read current data.yaml
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Show current path
    print(f"Current path in data.yaml: {data_config.get('path', 'NOT SET')}")
    
    # Update path to Colab location (absolute path)
    data_config['path'] = '/content/pallet_v1'
    
    # Write back
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
    
    # Verify the update
    print("✓ Updated data.yaml for Colab paths")
    print(f"  New path: {data_config['path']}")
    print(f"  Train: {data_config.get('train', 'NOT SET')}")
    print(f"  Val: {data_config.get('val', 'NOT SET')}")
    
    # Verify images exist
    train_path = Path(data_config['path']) / data_config.get('train', 'images/train')
    val_path = Path(data_config['path']) / data_config.get('val', 'images/val')
    
    if train_path.exists():
        print(f"  ✓ Train images found: {len(list(train_path.glob('*.*')))} files")
    else:
        print(f"  ⚠ Train images NOT found at: {train_path}")
    
    if val_path.exists():
        print(f"  ✓ Val images found: {len(list(val_path.glob('*.*')))} files")
    else:
        print(f"  ⚠ Val images NOT found at: {val_path}")
        
else:
    print(f"❌ data.yaml not found at {data_yaml_path}")
    print("  Make sure you uploaded/copied the dataset first!")
    print("  Expected location: /content/pallet_v1/data.yaml")

## Step 5: Train Model

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Configuration
model_name = "yolov8s.pt"
data_yaml = "/content/pallet_v1/data.yaml"
img_size = 960
epochs = 80
batch_size = 16

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Verify dataset structure
print(f"\n=== Dataset Verification ===")
if not Path(data_yaml).exists():
    print(f"❌ ERROR: data.yaml not found at {data_yaml}")
    print("Make sure you:")
    print("  1. Uploaded/copied the dataset")
    print("  2. Ran the previous cell to fix data.yaml paths")
else:
    print(f"✓ data.yaml found")
    
    # Double-check the paths in data.yaml
    import yaml
    with open(data_yaml, 'r') as f:
        config = yaml.safe_load(f)
    
    base_path = Path(config['path'])
    train_path = base_path / config.get('train', 'images/train')
    val_path = base_path / config.get('val', 'images/val')
    
    print(f"  Base path: {base_path}")
    print(f"  Train path: {train_path} - {'✓ EXISTS' if train_path.exists() else '❌ MISSING'}")
    print(f"  Val path: {val_path} - {'✓ EXISTS' if val_path.exists() else '❌ MISSING'}")
    
    if train_path.exists() and val_path.exists():
        print(f"\n✓ Starting training...")
        print(f"Model: {model_name}")
        print(f"Epochs: {epochs}")
        print(f"Batch size: {batch_size}")
        print(f"Image size: {img_size}")
        
        model = YOLO(model_name)
        results = model.train(
            data=data_yaml,
            imgsz=img_size,
            epochs=epochs,
            batch=batch_size,
            device=device,
            name="pallet_baseline_colab",
            project="/content/runs/detect",
            save=True,
            plots=True,
            verbose=True,
        )
        print("\n✓ Training complete!")
        print(f"Model: /content/runs/detect/pallet_baseline_colab/weights/best.pt")
    else:
        print(f"\n❌ Dataset structure incorrect!")
        print(f"Please check that images are in:")
        print(f"  {train_path}")
        print(f"  {val_path}")

## Step 6: Download Model

In [ ]:
# Download trained model
from google.colab import files

model_path = "/content/runs/detect/pallet_baseline_colab/weights/best.pt"
if Path(model_path).exists():
    files.download(model_path)
    print("✓ Model downloaded!")
else:
    print("❌ Model not found")
    print("Make sure training completed successfully.")